# CUDA Q4.4 exp approximation variants

Compare the three CUDA device helpers from `mrcp_quant/optimized_layers/common/tr_math.cuh` against a floating-point `exp(x)` baseline. The extension exposes a small test kernel through `trptq_gelu.exp_variant_u8_from_q44(...)` so the measurements use the actual CUDA code path.


In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f"Drive mount skipped: {exc}")


In [ ]:
import os
import sys
from pathlib import Path

import torch

PROJECT_DIR_PATH = globals().get("PROJECT_DIR_PATH", "/content/mrcp-tr-ptq")
PROJECT_DIR = Path(PROJECT_DIR_PATH).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

%cd {PROJECT_DIR_PATH}

TR_A_MIN = -8
TR_A_MAX = 0
EXP_SCALE_U8 = 255
GELU_EXT_DIR = PROJECT_DIR / "mrcp_quant" / "optimized_layers" / "gelu"


In [ ]:
GELU_EXT_DIR

In [ ]:
# # Build/install after editing CUDA sources. Rerun this cell whenever gelu_cuda*.{cpp,cu}
# # or common/tr_math.cuh changes, then restart/reload the notebook kernel if imports stay stale.
# !pip install -v --no-build-isolation -e "{GELU_EXT_DIR}"

import sys, shutil
from pathlib import Path

gelu_dir = Path(GELU_EXT_DIR)

shutil.rmtree(gelu_dir / "build", ignore_errors=True)

for p in gelu_dir.glob("_trptq_gelu*.so"):
    p.unlink()

!{sys.executable} -m pip install -v --no-build-isolation --no-cache-dir -e "{GELU_EXT_DIR}"

import _trptq_gelu
print(_trptq_gelu.__file__)


In [ ]:
import importlib

import trptq_gelu
import _trptq_gelu

# If you rebuilt in this same notebook session, reload the Python shim. For binary ABI changes,
# a kernel restart is still the cleanest option.
trptq_gelu = importlib.reload(trptq_gelu)
print("Loaded:", trptq_gelu.__file__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())

if not hasattr(_trptq_gelu, "exp_variant_arith"):
    raise RuntimeError(
        "The loaded _trptq_gelu binary is stale and does not export exp_variant_arith. "
        "Run the install cell again, then restart this notebook kernel/runtime before continuing."
    )
print("exp_variant_arith export: OK")


In [ ]:
def build_exp_lut_u8(device="cuda"):
    anchors = torch.arange(TR_A_MIN, TR_A_MAX + 1, dtype=torch.float32, device=device)
    lut = torch.exp(anchors)
    return torch.round(lut / lut.max() * EXP_SCALE_U8).to(torch.uint8)


def baseline_exp_u8_from_q44(x_q44):
    x = x_q44.to(torch.float32) / 16.0
    return torch.round(torch.exp(x) * EXP_SCALE_U8).clamp(0, 255).to(torch.uint8)


def cuda_exp_variants(x_q44, lut_u8):
    if x_q44.dtype != torch.int8:
        x_q44 = x_q44.to(torch.int8)
    if not x_q44.is_cuda:
        x_q44 = x_q44.cuda()
    return {
        "CUDA 0rd LUT anchor": trptq_gelu.exp_variant_u8_from_q44(x_q44, 0, exp_lut=lut_u8),
        "CUDA 1rd linear": trptq_gelu.exp_variant_u8_from_q44(x_q44, 1, exp_lut=lut_u8),
        "CUDA 2rd": trptq_gelu.exp_variant_u8_from_q44(x_q44, 2, exp_lut=lut_u8),
    }


def summarize(name, approx_u8, x_q44, baseline_u8):
    approx_cpu = approx_u8.detach().cpu()
    x_cpu = x_q44.detach().cpu().to(torch.int16)
    baseline_cpu = baseline_u8.detach().cpu()
    approx_f = approx_cpu.float() / EXP_SCALE_U8
    true_f = torch.exp(x_cpu.float() / 16.0)
    diff_u8 = approx_cpu.to(torch.int32) - baseline_cpu.to(torch.int32)
    diff_true = approx_f - true_f
    return {
        "name": name,
        "max_abs_u8": diff_u8.abs().max().item(),
        "mean_abs_u8": diff_u8.abs().float().mean().item(),
        "max_abs_float_vs_true": diff_true.abs().max().item(),
        "mean_abs_float_vs_true": diff_true.abs().mean().item(),
        "rmse_float_vs_true": diff_true.square().mean().sqrt().item(),
    }


def print_summary(rows):
    print(f"{'variant':<24} {'max_u8':>8} {'mean_u8':>10} {'max_true':>12} {'mean_true':>12} {'rmse_true':>12}")
    print("-" * 84)
    for row in rows:
        print(
            f"{row['name']:<24} {row['max_abs_u8']:8.0f} {row['mean_abs_u8']:10.4f} "
            f"{row['max_abs_float_vs_true']:12.4e} {row['mean_abs_float_vs_true']:12.4e} "
            f"{row['rmse_float_vs_true']:12.4e}"
        )


In [ ]:
assert torch.cuda.is_available(), "This notebook uses the CUDA extension; select a CUDA runtime first."
device = torch.device("cuda")
lut = trptq_gelu.build_exp_lut_u8(device=device)
print("LUT:", lut.detach().cpu().tolist())


In [ ]:
# Full signed int8 Q4.4 domain: [-8.0, 7.9375]. Positive inputs saturate because the
# LUT anchors clamp to [-8, 0]. GELU/sigmoid internals mostly use non-positive inputs.
x_q44 = torch.arange(-128, 128, device=device, dtype=torch.int16).to(torch.int8)
baseline_u8 = baseline_exp_u8_from_q44(x_q44).cpu()
variants = cuda_exp_variants(x_q44, lut)
rows = [summarize(name, value, x_q44, baseline_u8) for name, value in variants.items()]
print_summary(rows)


In [ ]:
# Non-positive domain used by sigmoid/GELU internals.
x_q44 = torch.arange(-128, 1, device=device, dtype=torch.int16).to(torch.int8)
baseline_u8 = baseline_exp_u8_from_q44(x_q44).cpu()
variants = cuda_exp_variants(x_q44, lut)
rows = [summarize(name, value, x_q44, baseline_u8) for name, value in variants.items()]
print_summary(rows)


In [ ]:
# Detailed table around the non-positive domain.
x_q44 = torch.arange(-128, 1, device=device, dtype=torch.int16).to(torch.int8)
baseline_u8 = baseline_exp_u8_from_q44(x_q44).cpu()
variants = {name: value.cpu() for name, value in cuda_exp_variants(x_q44, lut).items()}
x_cpu = x_q44.cpu().to(torch.int16)

indices = list(range(0, len(x_cpu), 8))
if indices[-1] != len(x_cpu) - 1:
    indices.append(len(x_cpu) - 1)

print(f"{'x':>8} {'base':>5} {'0rd':>5} {'1rd':>5} {'2rd':>5}")
print("-" * 36)
for i in indices:
    print(
        f"{(x_cpu[i].item() / 16.0):8.4f} {int(baseline_u8[i]):5d} "
        f"{int(variants['CUDA 0rd LUT anchor'][i]):5d} "
        f"{int(variants['CUDA 1rd linear'][i]):5d} "
        f"{int(variants['CUDA 2rd'][i]):5d}"
    )


In [ ]:
# Optional CUDA kernel timing on a large tensor.
x_big = torch.randint(-128, 1, (1_000_000,), device=device, dtype=torch.int8)

for name, variant in [
    ("CUDA 0rd LUT anchor", 0),
    ("CUDA 1rd linear", 1),
    ("CUDA 2rd", 2),
]:
    for _ in range(10):
        trptq_gelu.exp_variant_u8_from_q44(x_big, variant, exp_lut=lut)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(100):
        trptq_gelu.exp_variant_u8_from_q44(x_big, variant, exp_lut=lut)
    end.record()
    torch.cuda.synchronize()
    print(f"{name:<24} {start.elapsed_time(end) / 100:8.4f} ms")
